# 1.02 — Functions: the minimum — solutions

Lesson: [`lessons/1.02_functions_basics.md`](../lessons/1.02_functions_basics.md)

> **Reference solutions.** Open only after you have solved the problem yourself or climbed the hint ladder. Compare, don't copy: then re-solve from blank in a few days.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


## Problem 1 — temperature

`celsius_to_fahrenheit(c)` returns F = C × 9/5 + 32.

In [ ]:
def celsius_to_fahrenheit(c):
    return c * 9 / 5 + 32

In [ ]:
check(celsius_to_fahrenheit, [((0,), 32.0), ((100,), 212.0), ((-40,), -40.0), ((37,), 98.6)])

## Problem 2 — profit margin

`profit_margin(revenue, cost)` returns the margin as a **percentage** rounded to 1 decimal: (revenue − cost) / revenue × 100.

In [ ]:
def profit_margin(revenue, cost):
    return round((revenue - cost) / revenue * 100, 1)

In [ ]:
check(profit_margin, [((1000, 750), 25.0), ((250_000, 180_000), 28.0), ((3, 2), 33.3)])

## Problem 3 — print vs return

Run this cell and explain the output in a comment. Then write `shout(text)` that **returns** the text upper-cased with `"!"` added.

In [ ]:
def shout_print(text):
    print(text.upper() + "!")

result = shout_print("hello")
print("result is:", result)

In [ ]:
# Explanation: shout_print displays the text, but has no return statement, so it returns None.

def shout(text):
    return text.upper() + "!"

In [ ]:
check(shout, [(("hello",), "HELLO!"), (("",), "!")])

## Problem 4 — functions calling functions

Write two functions:

- `gross_to_net(gross, tax_rate)` → gross after tax (tax_rate is a fraction).
- `annual_net(monthly_gross, tax_rate)` → net pay for a year. It **must call** `gross_to_net`.

Round `annual_net`'s result to 2 decimals.

In [ ]:
def gross_to_net(gross, tax_rate):
    return gross * (1 - tax_rate)

def annual_net(monthly_gross, tax_rate):
    return round(gross_to_net(monthly_gross, tax_rate) * 12, 2)

In [ ]:
check(gross_to_net, [((100_000, 0.1), 90_000.0)])
check(annual_net, [((250_000, 0.12), 2_640_000.0), ((100_000, 0.0), 1_200_000.0)])

## Problem 5 — loan payment

Write `monthly_payment(principal, annual_rate, years)` using the standard formula (see the lesson), rounded to 2 decimals. Add a one-line docstring.

$$ \text{payment} = P \cdot \frac{r}{1 - (1 + r)^{-n}}, \quad r = \frac{\text{annual rate}}{12},\; n = 12 \times \text{years} $$

In [ ]:
def monthly_payment(principal, annual_rate, years):
    """Amortised monthly payment for a fixed-rate loan."""
    r = annual_rate / 12
    n = years * 12
    return round(principal * r / (1 - (1 + r) ** -n), 2)

In [ ]:
check(monthly_payment, [((1_000_000, 0.12, 5), 22244.45), ((500_000, 0.10, 3), 16133.59)])
expect("has a docstring", lambda: bool(monthly_payment.__doc__))

## Problem 6 ★ — total loan cost

`total_interest(principal, annual_rate, years)` returns how much interest is paid over the whole loan (all payments minus the principal), rounded to 2 decimals. Reuse `monthly_payment`.

In [ ]:
def total_interest(principal, annual_rate, years):
    payments = monthly_payment(principal, annual_rate, years) * years * 12
    return round(payments - principal, 2)

In [ ]:
check(total_interest, [((1_000_000, 0.12, 5), 334667.0), ((500_000, 0.10, 3), 80809.24)])